# Apple annual revenue and net income

Extract Apple's revenue and net income from SEC companyfacts for fiscal years 2021-2025, then save data/apple_financials.csv. The selected concepts are RevenueFromContractWithCustomerExcludingAssessedTax and NetIncomeLoss, using USD records.

Select annual 10-K facts, remove repeated comparative disclosures, and check that revenue and net income cover the same period. This is a retrospective extract using the latest eligible filing records, not a dataset reconstructed at historical forecast dates.

This notebook makes live SEC requests when run. Configure SEC_CONTACT_EMAIL locally. Investigate changes using the filing text and notes rather than attributing causes from these two series alone.


In [ ]:
from getpass import getpass
from pathlib import Path
import os
import time

import pandas as pd
import requests
from IPython.display import display
from dotenv import load_dotenv


current_path = Path.cwd().resolve()
candidate_roots = [current_path, *current_path.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "notebooks").is_dir()
        and (path / "data").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not find the project root containing notebooks and data."
    )

notebook_output_path = (
    project_root
    / "notebooks"
    / "08_sec_apple_metric.ipynb"
)
csv_output_path = (
    project_root
    / "data"
    / "apple_financials.csv"
)

print("project_root:", project_root)
print("notebook_output_path:", notebook_output_path)
print("csv_output_path:", csv_output_path)

In [ ]:
APPLE_TICKER = "AAPL"
apple_cik_int = 320193
apple_cik = f"{apple_cik_int:010d}"

SEC_COMPANYFACTS_URL_TEMPLATE = (
    "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
)
REQUEST_TIMEOUT_SECONDS = 30
REQUEST_INTERVAL_SECONDS = 0.2

FISCAL_YEAR_START = 2021
FISCAL_YEAR_END = 2025

REVENUE_CONCEPT_ID = (
    "RevenueFromContractWithCustomerExcludingAssessedTax"
)
NET_INCOME_CONCEPT_ID = "NetIncomeLoss"

print("ticker:", APPLE_TICKER)
print("apple_cik_int:", apple_cik_int)
print("apple_cik:", apple_cik)
print(
    "fiscal_year_range:",
    FISCAL_YEAR_START,
    "to",
    FISCAL_YEAR_END,
)
print("request_timeout_seconds:", REQUEST_TIMEOUT_SECONDS)
print("request_interval_seconds:", REQUEST_INTERVAL_SECONDS)

In [ ]:
load_dotenv(project_root / ".env", override=False)
sec_contact_email = os.getenv("SEC_CONTACT_EMAIL", "").strip()
if not sec_contact_email or "@" not in sec_contact_email:
    sec_contact_email = getpass("Contact email for the SEC User-Agent: ").strip()
if not sec_contact_email or "@" not in sec_contact_email:
    raise ValueError("Provide a valid contact email for the SEC User-Agent.")

sec_headers = {
    "User-Agent": f"U.S. Financial Research Project {sec_contact_email}",
    "Accept-Encoding": "gzip, deflate",
}


In [ ]:
def get_sec_json(
        url,
        headers,
        pause_seconds=REQUEST_INTERVAL_SECONDS,
):
    if pause_seconds < 0:
        raise ValueError("pause_seconds must be nonnegative.")

    time.sleep(pause_seconds)

    response = requests.get(
        url,
        headers=headers,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()

    payload = response.json()

    if not isinstance(payload, dict):
        raise TypeError("The SEC response must be a JSON object.")

    return payload

In [ ]:
companyfacts_url = SEC_COMPANYFACTS_URL_TEMPLATE.format(
    cik=apple_cik
)

apple_companyfacts_json = get_sec_json(
    url=companyfacts_url,
    headers=sec_headers,
)

required_top_level_keys = {
    "cik",
    "entityName",
    "facts",
}
missing_top_level_keys = (
    required_top_level_keys
    - set(apple_companyfacts_json.keys())
)

if missing_top_level_keys:
    raise ValueError(
        "The companyfacts response is missing required keys: "
        f"{sorted(missing_top_level_keys)}"
    )

if int(apple_companyfacts_json["cik"]) != apple_cik_int:
    raise ValueError("The companyfacts CIK does not match Apple.")

print("companyfacts_cik:", apple_companyfacts_json["cik"])
print(
    "companyfacts_entity_name:",
    apple_companyfacts_json["entityName"],
)
print(
    "companyfacts_top_level_keys:",
    sorted(apple_companyfacts_json.keys())
)

In [ ]:
facts_by_taxonomy = apple_companyfacts_json["facts"]

if "us-gaap" not in facts_by_taxonomy:
    raise KeyError("The companyfacts response has no us-gaap taxonomy.")

us_gaap_facts = facts_by_taxonomy["us-gaap"]

print("taxonomy_names:", sorted(facts_by_taxonomy.keys()))
print("us_gaap_concept_count:", len(us_gaap_facts))

In [ ]:
def find_concept_candidates(
        facts,
        keywords,
):
    normalized_keywords = tuple(
        keyword.lower()
        for keyword in keywords
    )
    candidate_rows = []

    for concept_id, concept_data in facts.items():
        concept_text = " ".join(
            [
                concept_id,
                str(concept_data.get("label", ""))
            ]
        ).lower()

        if not any(
            keyword in concept_text
            for keyword in normalized_keywords
        ):
            continue

        candidate_rows.append(
            {
                "concept_id": concept_id,
                "label": concept_data.get("label"),
                "unit_names": ", ".join(
                    sorted(
                        concept_data
                        .get("units", {})
                        .keys()
                    )
                ),
                "description": concept_data.get("description"),
            }
        )

    candidate_columns = [
        "concept_id",
        "label",
        "unit_names",
        "description"
    ]
    candidate_df = pd.DataFrame(
        candidate_rows,
        columns=candidate_columns,
    )

    return (
        candidate_df
        .sort_values("concept_id")
        .reset_index(drop=True)
    )

In [ ]:
revenue_candidate_df = find_concept_candidates(
    facts=us_gaap_facts,
    keywords=("Revenue", "Sales"),
)

print(
    "revenue_candidate_count:",
    len(revenue_candidate_df),
)
display(
    revenue_candidate_df[
        ["concept_id", "label", "unit_names"]
    ].head(30)
)

In [ ]:
net_income_candidate_df = find_concept_candidates(
    facts=us_gaap_facts,
    keywords=("NetIncome", "ProfitLoss"),
)

print(
    "net_income_candidate_count:",
    len(net_income_candidate_df),
)
display(
    net_income_candidate_df[
        ["concept_id", "label", "unit_names"]
    ].head(30)
)

In [ ]:
metric_concepts = {
    "revenue": REVENUE_CONCEPT_ID,
    "net_income": NET_INCOME_CONCEPT_ID,
}

concept_summary_rows = []

for metric_name, concept_id in metric_concepts.items():
    if concept_id not in us_gaap_facts:
        raise KeyError(
            f"The us-gaap taxonomy has no {concept_id} concept."
        )

    concept_data = us_gaap_facts[concept_id]
    unit_names = sorted(
        concept_data.get("units", {}).keys()
    )

    if "USD" not in unit_names:
        raise KeyError(
            f"The {concept_id} concept has no USD records."
        )

    concept_summary_rows.append(
        {
            "metric_name": metric_name,
            "concept_id": concept_id,
            "label": concept_data.get("label"),
            "unit_names": ", ".join(unit_names),
            "description": concept_data.get("description"),
        }
    )

concept_summary_df = pd.DataFrame(concept_summary_rows)

display(concept_summary_df)

## Selected concepts

- Consolidated revenue: us-gaap / RevenueFromContractWithCustomerExcludingAssessedTax.
- Consolidated net income: us-gaap / NetIncomeLoss.
- Both metrics use the USD records and retain filing and reporting-period metadata.


In [ ]:
def concept_usd_facts_to_df(
        companyfacts_json,
        concept_id,
):
    try:
        concept_data = (
            companyfacts_json["facts"]
            ["us-gaap"]
            [concept_id]
        )
        usd_records = concept_data["units"]["USD"]
    except KeyError as error:
        raise KeyError(
            f"Could not find USD facts for {concept_id}."
        ) from error

    facts_df = pd.DataFrame(usd_records)

    required_columns = [
        "start",
        "end",
        "val",
        "accn",
        "fy",
        "fp",
        "form",
        "filed",
    ]
    missing_columns = [
        column
        for column in required_columns
        if column not in facts_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"The {concept_id} facts are missing required columns: "
            f"{missing_columns}"
        )

    facts_df["start"] = pd.to_datetime(
        facts_df["start"],
        errors="coerce",
    )
    facts_df["end"] = pd.to_datetime(
        facts_df["end"],
        errors="coerce",
    )
    facts_df["filed"] = pd.to_datetime(
        facts_df["filed"],
        errors="coerce",
    )
    facts_df["val"] = pd.to_numeric(
        facts_df["val"],
        errors="coerce",
    )

    facts_df["duration_days"] = (
        facts_df["end"]
        - facts_df["start"]
    ).dt.days + 1
    facts_df["concept_id"] = concept_id
    facts_df["unit"] = "USD"
    return facts_df

In [ ]:
revenue_facts_df = concept_usd_facts_to_df(
    companyfacts_json=apple_companyfacts_json,
    concept_id=REVENUE_CONCEPT_ID,
)
net_income_facts_df = concept_usd_facts_to_df(
    companyfacts_json=apple_companyfacts_json,
    concept_id=NET_INCOME_CONCEPT_ID,
)

print("revenue_raw_fact_count:", len(revenue_facts_df))
print(
    "net_income_raw_fact_count:",
    len(net_income_facts_df),
)
print(
    "revenue_missing_value_count:",
    revenue_facts_df["val"].isna().sum(),
)
print(
    "net_income_missing_value_count:",
    net_income_facts_df["val"].isna().sum(),
)

In [ ]:
annual_candidate_columns = [
    "start",
    "end",
    "val",
    "duration_days",
    "form",
    "fy",
    "fp",
    "filed",
    "accn",
    "frame",
]

available_revenue_columns = [
    column
    for column in annual_candidate_columns
    if column in revenue_facts_df.columns
]

revenue_10k_fy_candidate_df = (
    revenue_facts_df.loc[
        revenue_facts_df["form"].eq("10-K")
        & revenue_facts_df["fp"].eq("FY")
    ]
    .sort_values(
        ["end", "filed"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print(
    "revenue_10k_fy_candidate_count:",
    len(revenue_10k_fy_candidate_df),
)
display(
    revenue_10k_fy_candidate_df[
        available_revenue_columns
    ].head(20)
)

In [ ]:
def extract_annual_usd_facts(
        companyfacts_json,
        concept_id,
        metric_name,
        fiscal_year_start,
        fiscal_year_end,
):
    facts_df = concept_usd_facts_to_df(
        companyfacts_json=companyfacts_json,
        concept_id=concept_id,
    )

    annual_df = facts_df.loc[
        facts_df["form"].eq("10-K")
        & facts_df["fp"].eq("FY")
        & facts_df["duration_days"].between(330, 400)
    ].copy()

    annual_df = annual_df.dropna(
        subset=["start", "end", "filed", "val"]
    )
    annual_df["fiscal_year"] = (
        annual_df["end"].dt.year
    )
    annual_df = annual_df.loc[
        annual_df["fiscal_year"].between(
            fiscal_year_start,
            fiscal_year_end,
        )
    ].copy()

    if annual_df.empty:
        raise ValueError(
            f"No annual {concept_id} facts were found in the selected range."
        )

    annual_df = (
        annual_df
        .sort_values(
            ["end", "filed", "accn"],
            ascending=[True, True, True],
        )
        .drop_duplicates(
            subset=["fiscal_year"],
            keep="last",
        )
        .sort_values("fiscal_year")
        .reset_index(drop=True)
    )

    annual_df = annual_df.rename(
        columns={
            "start": f"{metric_name}_period_start",
            "end": f"{metric_name}_period_end",
            "val": f"{metric_name}_usd",
            "filed": f"{metric_name}_filed",
            "accn": f"{metric_name}_accn",
            "duration_days": f"{metric_name}_duration_days",
            "concept_id": f"{metric_name}_concept",
        }
    )

    output_columns = [
        "fiscal_year",
        f"{metric_name}_period_start",
        f"{metric_name}_period_end",
        f"{metric_name}_usd",
        f"{metric_name}_filed",
        f"{metric_name}_accn",
        f"{metric_name}_duration_days",
        f"{metric_name}_concept",
    ]

    return annual_df[output_columns]

In [ ]:
revenue_annual_df = extract_annual_usd_facts(
    companyfacts_json=apple_companyfacts_json,
    concept_id=REVENUE_CONCEPT_ID,
    metric_name="revenue",
    fiscal_year_start=FISCAL_YEAR_START,
    fiscal_year_end=FISCAL_YEAR_END,
)
net_income_annual_df = extract_annual_usd_facts(
    companyfacts_json=apple_companyfacts_json,
    concept_id=NET_INCOME_CONCEPT_ID,
    metric_name="net_income",
    fiscal_year_start=FISCAL_YEAR_START,
    fiscal_year_end=FISCAL_YEAR_END,
)

print("revenue_annual_row_count:", len(revenue_annual_df))
print(
    "net_income_annual_row_count:",
    len(net_income_annual_df),
)

display(revenue_annual_df)
display(net_income_annual_df)

In [ ]:
merged_financials_df = revenue_annual_df.merge(
    net_income_annual_df,
    on="fiscal_year",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

if not merged_financials_df["_merge"].eq("both").all():
    display(
        merged_financials_df.loc[
            merged_financials_df["_merge"].ne("both")
        ]
    )
    raise ValueError(
        "Revenue and net income cover different fiscal years."
    )

merged_financials_df = merged_financials_df.drop(
    columns="_merge"
)

print("merged_row_count:", len(merged_financials_df))
display(merged_financials_df)

In [ ]:
period_match_mask = (
    merged_financials_df["revenue_period_start"].eq(
        merged_financials_df["net_income_period_start"]
    )
    & merged_financials_df["revenue_period_end"].eq(
        merged_financials_df["net_income_period_end"]
    )
)

if not period_match_mask.all():
    period_mismatch_df = merged_financials_df.loc[
        ~period_match_mask,
        [
            "fiscal_year",
            "revenue_period_start",
            "revenue_period_end",
            "net_income_period_start",
            "net_income_period_end",
        ],
    ]
    display(period_mismatch_df)
    raise ValueError(
        "Revenue and net income cover different reporting periods."
    )

print("all_metric_periods_match:", True)

In [ ]:
apple_financials_df = merged_financials_df.copy()

apple_financials_df["company_name"] = (
    apple_companyfacts_json["entityName"]
)
apple_financials_df["ticker"] = APPLE_TICKER
apple_financials_df["cik"] = apple_cik
apple_financials_df["period_start"] = (
    apple_financials_df["revenue_period_start"]
)
apple_financials_df["period_end"] = (
    apple_financials_df["revenue_period_end"]
)
apple_financials_df["form"] = "10-K"
apple_financials_df["unit"] = "USD"
apple_financials_df["source"] = (
    "SEC EDGAR companyfacts"
)

apple_financials_df = apple_financials_df.drop(
    columns=[
        "revenue_period_start",
        "revenue_period_end",
        "net_income_period_start",
        "net_income_period_end",
    ]
)

final_columns = [
    "company_name",
    "ticker",
    "cik",
    "fiscal_year",
    "period_start",
    "period_end",
    "revenue_usd",
    "net_income_usd",
    "form",
    "unit",
    "revenue_concept",
    "net_income_concept",
    "revenue_duration_days",
    "net_income_duration_days",
    "revenue_filed",
    "net_income_filed",
    "revenue_accn",
    "net_income_accn",
    "source",
]

apple_financials_df = (
    apple_financials_df[final_columns]
    .sort_values("fiscal_year")
    .reset_index(drop=True)
)

print("final_shape:", apple_financials_df.shape)
display(apple_financials_df)

In [ ]:
expected_fiscal_year = set(
    range(
        FISCAL_YEAR_START,
        FISCAL_YEAR_END + 1 ,
    )
)
actual_fiscal_years = set(
    apple_financials_df["fiscal_year"].tolist()
)
missing_fiscal_years = sorted(
    expected_fiscal_year - actual_fiscal_years
)
unexpected_fiscal_years = sorted(
    actual_fiscal_years - expected_fiscal_year
)

print("missing_fiscal_years:", missing_fiscal_years)
print("unexpected_fiscal_years:", unexpected_fiscal_years)
print(
    "duplicate_fiscal_year_count:",
    apple_financials_df["fiscal_year"]
    .duplicated()
    .sum()
)
print(
    "missing_value_count:",
    apple_financials_df[
        ["revenue_usd", "net_income_usd"]
    ]
    .isna()
    .sum()
    .to_dict(),
)

if missing_fiscal_years or unexpected_fiscal_years:
    raise ValueError("The fiscal-year range does not match the selected range.")

if apple_financials_df["fiscal_year"].duplicated().any():
    raise ValueError("Duplicate fiscal years were found.")

if apple_financials_df[
    ["revenue_usd", "net_income_usd"]
].isna().any().any():
    raise ValueError("Revenue or net income contains missing values.")


In [ ]:
display_financials_df = (
    apple_financials_df[
        [
            "fiscal_year",
            "period_start",
            "period_end",
            "revenue_usd",
            "net_income_usd",
        ]
    ]
    .copy()
)

display_financials_df["revenue_usd_b"] = (
    display_financials_df["revenue_usd"]
    / 1_000_000_000
)

display_financials_df["net_income_usd_b"] = (
    display_financials_df["net_income_usd"]
    / 1_000_000_000
)

print("display_unit: USD billion")
display(
    display_financials_df[
        [
            "fiscal_year",
            "period_start",
            "period_end",
            "revenue_usd_b",
            "net_income_usd_b",
        ]
    ].round(3)
)

In [ ]:
# Reference values from the teaching snapshot; later filings may revise them.
reference_financials_df = pd.DataFrame(
    {
        "fiscal_year": [2021, 2022, 2023, 2024, 2025],
        "reference_revenue_usd": [
            365_817_000_000,
            394_328_000_000,
            383_285_000_000,
            391_035_000_000,
            416_161_000_000,
        ],
        "reference_net_income_usd": [
            94_680_000_000,
            99_803_000_000,
            96_995_000_000,
            93_736_000_000,
            112_010_000_000,
        ],
    }
)

reference_check_df = apple_financials_df.merge(
    reference_financials_df,
    on="fiscal_year",
    how="left",
    validate="one_to_one",
)

reference_check_df["revenue_matches_reference"] = (
    reference_check_df["revenue_usd"].eq(
        reference_check_df["reference_revenue_usd"]
    )
)
reference_check_df["net_income_matches_reference"] = (
    reference_check_df["net_income_usd"].eq(
        reference_check_df["reference_net_income_usd"]
    )
)

display(
    reference_check_df[
        [
            "fiscal_year",
            "revenue_matches_reference",
            "net_income_matches_reference",
            "revenue_accn",
            "net_income_accn",
        ]
    ]
)

In [ ]:
if not csv_output_path.parent.is_dir():
    raise FileNotFoundError(
        "The data directory is missing. Check the output location."
    )

apple_financials_df.to_csv(
    csv_output_path,
    index=False,
    date_format="%Y-%m-%d",
)

print("saved_csv_path:", csv_output_path)
print("saved_csv_exists:", csv_output_path.exists())
print("saved_csv_size_bytes:", csv_output_path.stat().st_size)

In [ ]:
reloaded_financials_df = pd.read_csv(
    csv_output_path,
    dtype={
        "ticker": "string",
        "cik": "string",
    },
    parse_dates=[
        "period_start",
        "period_end",
        "revenue_filed",
        "net_income_filed",
    ],
)

print(
    "reloaded_shape:",
    reloaded_financials_df.shape,
)
print(
    "reloaded_cik_values:",
    reloaded_financials_df["cik"].unique().tolist(),
)
print(
    "reloaded_fiscal_years:",
    reloaded_financials_df["fiscal_year"].tolist(),
)

display(reloaded_financials_df)

In [ ]:
if list(reloaded_financials_df.columns) != final_columns:
    raise ValueError("The CSV column order does not match the output schema.")

if len(reloaded_financials_df) != 5:
    raise ValueError("The CSV does not contain the five selected fiscal years.")

if not reloaded_financials_df["cik"].eq(apple_cik).all():
    raise ValueError("The CSV contains an unexpected Apple CIK.")

if not reloaded_financials_df["ticker"].eq(APPLE_TICKER).all():
    raise ValueError("The CSV contains a ticker other than AAPL.")

if not reloaded_financials_df["form"].eq("10-K").all():
    raise ValueError("The CSV contains a form other than 10-K.")

if not reloaded_financials_df["unit"].eq("USD").all():
    raise ValueError("The CSV contains a unit other than USD.")

print("csv_validation_passed:", True)

In [ ]:
first_row = apple_financials_df.iloc[0]
last_row = apple_financials_df.iloc[-1]

revenue_change_usd_b = (
    last_row["revenue_usd"] - first_row["revenue_usd"]
) / 1_000_000_000
net_income_change_usd_b = (
    last_row["net_income_usd"] - first_row["net_income_usd"]
) / 1_000_000_000

summary_sentence = (
    f"Apple revenue was USD {first_row['revenue_usd'] / 1_000_000_000:.3f} billion "
    f"in fiscal {int(first_row['fiscal_year'])} and "
    f"USD {last_row['revenue_usd'] / 1_000_000_000:.3f} billion "
    f"in fiscal {int(last_row['fiscal_year'])}, "
    f"a change of USD {revenue_change_usd_b:+.3f} billion. "
    f"Net income changed from USD {first_row['net_income_usd'] / 1_000_000_000:.3f} billion "
    f"to USD {last_row['net_income_usd'] / 1_000_000_000:.3f} billion "
    f"(USD {net_income_change_usd_b:+.3f} billion). "
    "The table summarizes annual SEC companyfacts records. "
    "The filing text and notes are needed to investigate the reasons for these changes."
)

print(summary_sentence)
